# RSNA Knee - v5 C: submission (prediction only)

**Inputs:** the competition data + your trained model datasets (`knee-v5-convnext`, and `knee-v5-effnet` if
you trained a second family). **Internet OFF**, GPU T4.

It reads each hidden test study once, with **exactly the image steps of notebook A** (settings come from
the trained models' `config.json`), runs every fold model, averages the folds of each family, then
averages the families by rank. The safety steps of our earlier submission cells are kept: a study that
cannot be read gets black pictures, a failed batch gets 0.5, and the file is checked before Kaggle reads it.

In [1]:
import sys
# Kaggle's newest image has a broken "triton" package, so "import timm" fails with
# "module 'triton' has no attribute 'backends'". PyTorch needs triton only for torch.compile,
# which we never use, so we always hide it (even if PyTorch is already loaded).
for _m in [m for m in sys.modules if m == "triton" or m.startswith("triton.")]:
    del sys.modules[_m]
sys.modules["triton"] = None
if "torch.utils._triton" in sys.modules:                 # forget an earlier "triton is there" answer
    for _f in ("has_triton_package", "has_triton"):
        _fn = getattr(sys.modules["torch.utils._triton"], _f, None)
        if hasattr(_fn, "cache_clear"):
            _fn.cache_clear()

import platform
import time as _time
NOTEBOOK_START = _time.time()
try:
    import torch
    import timm
    import torchvision
except Exception as e:
    raise RuntimeError(f"PyTorch / timm still fail to load ({type(e).__name__}: {e}). "
                       "Click Run -> Restart & clear cell outputs, then run all cells again.") from e
print(f"python {platform.python_version()} | torch {torch.__version__} | timm {timm.__version__} | "
      f"torchvision {torchvision.__version__} | triton hidden")

import os
import glob
import json
import time
import random

import cv2
import numpy as np
import pandas as pd
import pydicom

try:
    from pydicom.pixels import apply_voi_lut
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_voi_lut

COMP = next(p for p in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
                        "/kaggle/input/rsna-knee-abnormality-detection"] if os.path.exists(p))

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
PLANES = ["Sagittal", "Coronal", "Axial"]
print("data root:", COMP)

from torch.utils.data import Dataset, DataLoader

python 3.13.15 | torch 2.11.0+cu128 | timm 1.0.29 | torchvision 0.26.0+cu128 | triton hidden
data root: /kaggle/input/competitions/rsna-knee-abnormality-detection


In [2]:
def plane_blocks(prep):
    """Where each plane's slices sit inside one study's array."""
    blocks, start = {}, 0
    for plane in PLANES:
        n = prep["slices"][plane]
        blocks[plane] = (start, start + n)
        start += n
    return blocks


def pick_series(series_df):
    """One series per plane. Fluid-sensitive first (in this data it always equals fat-suppressed)."""
    series_df = series_df.copy()
    for col, empty in (("Fluid_Sensitive", 0), ("Fat_Suppression", 0), ("Anatomical_Plane", ""), ("SeriesInstanceUID", "")):
        if col not in series_df.columns:
            series_df[col] = empty                    # a missing column must not crash the hidden test run
    if "StudyInstanceUID" not in series_df.columns or series_df.empty:
        return pd.DataFrame(columns=["StudyInstanceUID"] + PLANES)
    ranked = series_df.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
    rows = []
    for study, group in ranked.groupby("StudyInstanceUID"):
        row = {"StudyInstanceUID": str(study)}
        for plane in PLANES:
            same = group[group["Anatomical_Plane"] == plane]
            row[plane] = str(same.iloc[0]["SeriesInstanceUID"]) if len(same) else ""
        rows.append(row)
    return pd.DataFrame(rows)


def slice_key(header):
    """Position along the body axis the stack runs along (x for sagittal, y for coronal, z for axial).
    Using the body axis, not the scanner's slice direction, gives the same order on every scanner."""
    try:
        pos = np.array(header.ImagePositionPatient, dtype=float)
        orient = np.array(header.ImageOrientationPatient, dtype=float)
        normal = np.cross(orient[:3], orient[3:])
        return float(pos[int(np.argmax(np.abs(normal)))])
    except Exception:
        return float(getattr(header, "InstanceNumber", 0) or 0)


def sorted_paths(folder):
    ordered = []
    for path in glob.glob(os.path.join(folder, "*.dcm")):
        try:
            header = pydicom.dcmread(path, stop_before_pixels=True,
                                     specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
            ordered.append((slice_key(header), path))
        except Exception:
            continue
    ordered.sort(key=lambda item: item[0])
    return [path for _, path in ordered]


def read_image(path):
    """One slice as floats, plus mm per pixel along rows and columns."""
    try:
        data = pydicom.dcmread(path)
        try:
            image = apply_voi_lut(data.pixel_array, data).astype(np.float32)
        except Exception:
            image = np.asarray(data.pixel_array, dtype=np.float32)
        image = np.nan_to_num(np.squeeze(image), nan=0.0, posinf=0.0, neginf=0.0)
        if image.ndim != 2:
            return None, None
        if getattr(data, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image
        spacing = getattr(data, "PixelSpacing", None)
        mm = (float(spacing[0]), float(spacing[1])) if spacing is not None and len(spacing) >= 2 else None
        return image, mm
    except Exception:
        return None, None


def crop_to_mm(image, mm, crop_mm):
    """Keep the same real area of knee, whatever the scanner's pixel size."""
    if not mm or mm[0] <= 0 or mm[1] <= 0:
        return image
    half_y, half_x = int(round(crop_mm / mm[0] / 2)), int(round(crop_mm / mm[1] / 2))
    mid_y, mid_x = image.shape[0] // 2, image.shape[1] // 2
    cropped = image[max(mid_y - half_y, 0):mid_y + half_y, max(mid_x - half_x, 0):mid_x + half_x]
    return cropped if cropped.size else image


def load_series(folder, n, prep):
    size = prep["img_size"]
    empty = np.zeros((n, size, size), dtype=np.uint8)
    paths = sorted_paths(folder)
    if not paths:
        return empty

    start = int(prep["span"][0] * len(paths))
    stop = max(int(prep["span"][1] * len(paths)), start + 1)
    picked = np.linspace(start, stop - 1, n).round().astype(int)

    slices, last_good = [], None
    for i in picked:
        image, mm = read_image(paths[i])
        if image is None:
            slices.append(last_good if last_good is not None else np.zeros((size, size), np.float32))
            continue
        image = cv2.resize(crop_to_mm(image, mm, prep["crop_mm"]), (size, size), interpolation=cv2.INTER_AREA)
        slices.append(image)
        last_good = image

    stack = np.stack(slices).astype(np.float32)
    low, high = np.percentile(stack, [1, 99])         # one window for the whole series
    stack = (np.clip(stack, low, high) - low) / max(high - low, 1e-6)
    return (stack * 255).astype(np.uint8)


def knee_side(study_uid, row, root):
    """'L', 'R' or None. Laterality tag first, else where the image centre sits (right knee at x < 0)."""
    tags, centres = [], []
    for plane in PLANES:
        series_uid = row.get(plane, "")
        if not isinstance(series_uid, str) or not series_uid:
            continue
        files = glob.glob(os.path.join(root, study_uid, series_uid, "*.dcm"))
        if not files:
            continue
        try:
            h = pydicom.dcmread(files[0], stop_before_pixels=True)
        except Exception:
            continue
        tag = str(getattr(h, "Laterality", "") or "").strip().upper()
        if tag[:1] in ("L", "R"):
            tags.append(tag[0])
        try:
            pos = np.array(h.ImagePositionPatient, dtype=float)
            orient = np.array(h.ImageOrientationPatient, dtype=float)
            spacing = np.array(h.PixelSpacing, dtype=float)
            centre = (pos + orient[:3] * spacing[1] * float(h.Columns) / 2
                          + orient[3:] * spacing[0] * float(h.Rows) / 2)
            centres.append(centre[0])
        except Exception:
            pass
    if tags:
        return tags[0]
    if centres and abs(float(np.median(centres))) >= 20:
        return "R" if float(np.median(centres)) < 0 else "L"
    return None


def load_study(study_uid, row, root, prep):
    """One study as (all slices, size, size) uint8: sagittal block, coronal block, axial block.
    Left knees are mirrored so every knee looks like a right knee. Missing planes stay black."""
    blocks = plane_blocks(prep)
    total = blocks[PLANES[-1]][1]
    volume = np.zeros((total, prep["img_size"], prep["img_size"]), dtype=np.uint8)

    for plane in PLANES:
        series_uid = row.get(plane, "")
        if isinstance(series_uid, str) and series_uid:
            a, b = blocks[plane]
            try:
                volume[a:b] = load_series(os.path.join(root, study_uid, series_uid), b - a, prep)
            except Exception:
                pass

    try:
        if knee_side(study_uid, row, root) == "L":
            a, b = blocks["Sagittal"]
            volume[a:b] = volume[a:b][::-1]                  # sagittal: read the stack backwards
            for plane in ("Coronal", "Axial"):
                a, b = blocks[plane]
                volume[a:b] = volume[a:b, :, ::-1]           # coronal / axial: flip left-right
    except Exception:
        pass
    return volume

In [3]:
import torch
import torch.nn as nn
import timm
import torchvision

# RadImageNet: CNNs pretrained on 1.35 million radiology images (CT, MRI, ultrasound), MIT licence
RAD_FILES = {"radimagenet_resnet50": "ResNet50.pt",
             "radimagenet_densenet121": "DenseNet121.pt",
             "radimagenet_inceptionv3": "InceptionV3.pt"}


class RadTrunk(nn.Module):
    """torchvision ResNet50 / DenseNet121 / InceptionV3 without the classifier."""

    def __init__(self, name):
        super().__init__()
        self.name = name
        if name == "radimagenet_resnet50":
            self.net = nn.Sequential(*list(torchvision.models.resnet50(weights=None).children())[:-2])
            self.num_features = 2048
        elif name == "radimagenet_densenet121":
            self.net = torchvision.models.densenet121(weights=None, memory_efficient=True).features
            self.num_features = 1024
        else:
            m = torchvision.models.inception_v3(weights=None, aux_logits=False, init_weights=False)
            m.fc, m.dropout = nn.Identity(), nn.Identity()
            self.net = m
            self.num_features = 2048

    def forward(self, x):
        y = self.net(x)
        if self.name == "radimagenet_densenet121":
            y = torch.relu(y)
        return y.mean(dim=(2, 3)) if y.ndim == 4 else y

    def set_grad_checkpointing(self, enable=True):
        pass        # DenseNet already saves memory; ResNet50 / InceptionV3 fit at batch 4


def find_rad_file(name):
    hits = sorted(glob.glob(f"/kaggle/input/datasets/*/*/{name}") + glob.glob(f"/kaggle/input/datasets/*/*/*/{name}"))
    if hits:
        return hits[0]
    from huggingface_hub import hf_hub_download        # needs Internet ON (training notebooks only)
    return hf_hub_download("Lab-Rasool/RadImageNet", name)


def load_rad_weights(trunk, path):
    """The files come in different key layouts, so tensors are matched in order and every shape is checked."""
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, nn.Module):
        obj = obj.state_dict()
    if isinstance(obj, dict) and "state_dict" in obj:
        obj = obj["state_dict"]
    skip = ("fc.", "classifier.", "AuxLogits.", "num_batches_tracked")
    src = [v for k, v in obj.items() if torch.is_tensor(v) and not any(t in k for t in skip)]
    dst = [k for k in trunk.net.state_dict() if not any(t in k for t in skip)]
    shapes_ok = len(src) == len(dst) and all(tuple(v.shape) == tuple(trunk.net.state_dict()[k].shape) for v, k in zip(src, dst))
    if not shapes_ok:
        raise ValueError(f"RadImageNet file {path} does not match {trunk.name}: {len(src)} tensors vs {len(dst)} expected")
    trunk.net.load_state_dict(dict(zip(dst, src)), strict=False)
    print(f"RadImageNet weights loaded: {path} ({len(dst)} tensors, all shapes match)")


def build_encoder(backbone, pretrained=False):
    """Returns the image network and the input scaling its weights expect."""
    if backbone in RAD_FILES:
        trunk = RadTrunk(backbone)
        if pretrained:
            load_rad_weights(trunk, find_rad_file(RAD_FILES[backbone]))
        return trunk, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)      # RadImageNet expects -1..1
    enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
    cfg = enc.pretrained_cfg
    return enc, cfg.get("mean", (0.485, 0.456, 0.406)), cfg.get("std", (0.229, 0.224, 0.225))


def window_table(prep):
    """Every sliding 3-slice picture: (plane id, centre slice, position in the plane)."""
    rows = []
    for p, plane in enumerate(PLANES):
        a, b = plane_blocks(prep)[plane]
        for c in range(a + 1, b - 1):
            rows.append((p, c, c - a))
    return np.array(rows, dtype=np.int64)


def make_windows(volume, chosen, table):
    """(len(chosen), 3, H, W) floats in 0-1, plus plane ids, positions and a valid flag per picture."""
    rows = table[chosen]
    centres = rows[:, 1]
    x = np.stack([volume[c - 1:c + 2] for c in centres]).astype(np.float32) / 255.0
    valid = x.reshape(len(rows), -1).max(axis=1) > 0
    return x, rows[:, 0], rows[:, 2], valid


class KneeNetV5(nn.Module):
    """Backbone per picture -> plane and position tags -> small transformer -> one attention per finding."""

    def __init__(self, backbone, pretrained=False, dim=256, max_pos=64):
        super().__init__()
        self.encoder, mean, std = build_encoder(backbone, pretrained)
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

        self.proj = nn.Linear(self.encoder.num_features, dim)
        self.plane_emb = nn.Embedding(len(PLANES), dim)
        self.pos_emb = nn.Embedding(max_pos, dim)
        layer = nn.TransformerEncoderLayer(dim, 4, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.mixer = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.Linear(dim, len(LABELS))
        self.out_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
        self.out_b = nn.Parameter(torch.zeros(len(LABELS)))

    def forward(self, x, plane, pos, valid):            # x: (batch, pictures, 3, H, W)
        b, w = x.shape[:2]
        feats = self.encoder((x.flatten(0, 1) - self.mean) / self.std).view(b, w, -1)
        h = self.proj(feats) + self.plane_emb(plane) + self.pos_emb(pos)

        valid = valid.clone()
        valid[valid.sum(1) == 0] = True                  # a study with no pictures: avoid empty attention
        h = self.norm(self.mixer(h, src_key_padding_mask=~valid))

        scores = self.attn(h).float().masked_fill(~valid[..., None], -1e4).softmax(dim=1)
        pooled = torch.einsum("bwk,bwd->bkd", scores, h.float())    # one summary per finding
        return (pooled * self.out_w).sum(-1) + self.out_b

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE != "cuda":
    print("warning: no GPU. Settings -> Accelerator -> GPU T4. On CPU the hidden test run may not finish in 9 hours")

# every attached folder with a config.json is one model family
families = {}
cfg_paths = set()
for folder, subdirs, files in os.walk("/kaggle/input"):
    if folder.startswith(COMP) or folder.count(os.sep) - "/kaggle/input".count(os.sep) >= 8:
        subdirs[:] = []                                   # skip the scans, and do not go too deep
        continue
    if "config.json" in files:
        cfg_paths.add(os.path.join(folder, "config.json"))
for cfg_path in sorted(cfg_paths):
    try:
        cfg = json.load(open(cfg_path))
    except Exception:
        continue
    if not isinstance(cfg, dict) or not {"family", "backbone", "prep", "labels"} <= set(cfg):
        continue                                           # some other dataset's config.json
    folds = sorted(glob.glob(os.path.join(os.path.dirname(cfg_path), "v5_fold*.pt")))
    if folds:
        fam = families.setdefault(cfg["family"], {"cfg": cfg, "folds": [], "dirs": []})
        fam["folds"] += folds
        fam["dirs"].append(os.path.dirname(cfg_path))
        fam.setdefault("folds_ids", set()).add(cfg.get("folds_id", "not saved"))
for name, fam in families.items():
    print(f"family {name}: {fam['cfg']['backbone']}, {len(fam['folds'])} fold models")
    if len(fam["folds_ids"] - {"not saved"}) > 1:
        print(f"  warning: the parts of {name} were trained with different folds {sorted(fam['folds_ids'])}; "
              "its check score is too optimistic (attach the same label tables in every training session)")
assert families, "Attach at least one trained model dataset (the output of a training notebook)."

prep = next(iter(families.values()))["cfg"]["prep"]
assert all(f["cfg"]["prep"] == prep for f in families.values()), "all families must use the same image settings"
TABLE = window_table(prep)

device: cuda
family rad_densenet121: radimagenet_densenet121, 1 fold models


## Choose the model families

Every trained family saved its out-of-fold predictions (each study predicted by a model that never saw
it). Here they are compared on the same studies:

1. score each family alone;
2. start from the best one;
3. add another family only if the rank average gets better by at least `MIN_GAIN`.

So a weak family is never mixed in (for another team, mixing in a weaker model lowered 0.943 to 0.932).
Set `SELECT` to a list to force your own choice.

In [4]:
SELECT = "auto"          # or e.g. ["convnext", "rad_resnet50"]
MIN_GAIN = 0.001
from sklearn.metrics import roc_auc_score


def macro_auc(t, p, w):
    scores = []
    for i in range(len(LABELS)):
        keep = (w[:, i] > 0.5) & np.isfinite(p[:, i])
        tt = (t[keep, i] > 0.5).astype(int)
        if len(np.unique(tt)) == 2:
            scores.append(roc_auc_score(tt, p[keep, i]))
    return float(np.mean(scores)) if scores else float("nan")


oofs = {}
for name, fam in families.items():
    files = [f for d in fam["dirs"] for f in sorted(glob.glob(os.path.join(d, "oof_fold*.csv")))]
    if files:
        df = pd.concat([pd.read_csv(f) for f in files])
        df["StudyInstanceUID"] = df["StudyInstanceUID"].astype(str)
        oofs[name] = df.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")

if SELECT != "auto":
    chosen = [n for n in SELECT if n in families]
    unknown = [n for n in SELECT if n not in families]
    if unknown:
        print(f"warning: SELECT names not found and ignored: {unknown}. Found: {list(families)}")
    if not chosen:
        print("warning: nothing in SELECT was found, so every family is used")
        chosen = list(families)
elif len(oofs) < 2:
    chosen = list(families)
else:
    common = sorted(set.intersection(*[set(d.index) for d in oofs.values()]))
if SELECT == "auto" and len(oofs) >= 2 and len(common) < 100:
    print(f"warning: the families share only {len(common)} check studies (trained on different folds?), "
          "so they cannot be compared; every family is used with equal weight")
    chosen = list(families)
elif SELECT == "auto" and len(oofs) >= 2:
    first = oofs[next(iter(oofs))].loc[common]
    t = first[["t_" + l for l in LABELS]].values
    w = first[["w_" + l for l in LABELS]].values
    ranks = {n: d.loc[common, LABELS].rank(pct=True).values for n, d in oofs.items()}
    alone = {n: macro_auc(t, r, w) for n, r in ranks.items()}
    for n, sc in sorted(alone.items(), key=lambda kv: -kv[1]):
        print(f"{n:22s} OOF alone {sc:.4f}   ({len(families[n]['folds'])} folds)")

    chosen = [max(alone, key=alone.get)]
    current = alone[chosen[0]]
    while True:
        gains = {n: macro_auc(t, np.mean([ranks[m] for m in chosen + [n]], axis=0), w)
                 for n in ranks if n not in chosen}
        if not gains:
            break
        n, sc = max(gains.items(), key=lambda kv: kv[1])
        if sc < current + MIN_GAIN:
            break
        print(f"add {n:18s} -> OOF {sc:.4f} (+{sc - current:.4f})")
        chosen.append(n)
        current = sc
    print(f"\nchosen: {chosen}   OOF of the blend: {current:.4f}   studies compared: {len(common)}")

# the same choice on the 58 gold studies, for information only
train = pd.read_csv(f"{COMP}/train.csv")
gold = train.dropna(subset=LABELS)
gold["StudyInstanceUID"] = gold["StudyInstanceUID"].astype(str)
gold_ranks = []
for n in chosen:
    files = [f for d in families[n]["dirs"] for f in sorted(glob.glob(os.path.join(d, "gold_fold*.csv")))]
    if files:
        g = pd.concat([pd.read_csv(f) for f in files])
        g["StudyInstanceUID"] = g["StudyInstanceUID"].astype(str)
        gold_ranks.append(g.groupby("StudyInstanceUID")[LABELS].mean().reindex(gold["StudyInstanceUID"]).rank(pct=True).values)
if gold_ranks:
    gw = np.ones((len(gold), len(LABELS)))
    print(f"gold score of the chosen blend: {macro_auc(gold[LABELS].values, np.mean(gold_ranks, axis=0), gw):.4f}")

families = {n: families[n] for n in chosen}

gold score of the chosen blend: 0.6831


/tmp/ipykernel_22/3688614269.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gold["StudyInstanceUID"] = gold["StudyInstanceUID"].astype(str)


In [5]:
test = pd.read_csv(f"{COMP}/test.csv")
assert "StudyInstanceUID" in test.columns, "test.csv has no StudyInstanceUID column"
test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)

series_path = f"{COMP}/test_series.csv"
test_series = pd.read_csv(series_path) if os.path.isfile(series_path) else pd.DataFrame()
for col in ("StudyInstanceUID", "SeriesInstanceUID"):
    if col in test_series.columns:
        test_series[col] = test_series[col].astype(str)
test_chosen = pick_series(test_series).set_index("StudyInstanceUID")

TEST_ROOT = f"{COMP}/test_series"
if not os.path.isdir(TEST_ROOT):
    print("warning: test_series folder not found; every study gets black pictures")

study_uids = list(dict.fromkeys(test["StudyInstanceUID"]))       # each study once, even if test.csv repeats it
print("rows in test.csv:", len(test), " different studies:", len(study_uids),
      " missing from test_series.csv:", len(set(study_uids) - set(test_chosen.index)))

rows in test.csv: 3  different studies: 3  missing from test_series.csv: 0


In [6]:
class TestSet(Dataset):
    def __init__(self, uids):
        self.uids = uids

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, k):
        uid = self.uids[k]
        try:
            row = test_chosen.loc[uid].to_dict() if uid in test_chosen.index else {}
            volume = load_study(uid, row, TEST_ROOT, prep)
        except Exception as e:
            # one bad study must not stop the submission
            print(f"warning: black pictures for study {uid}: {type(e).__name__}: {e}")
            volume = np.zeros((len(TABLE) + 2 * len(PLANES), prep["img_size"], prep["img_size"]), np.uint8)
        x, plane, pos, valid = make_windows(volume, np.arange(len(TABLE)), TABLE)
        return torch.from_numpy(x), torch.from_numpy(plane), torch.from_numpy(pos), torch.from_numpy(valid), uid


def worker_init(worker_id):
    cv2.setNumThreads(0)          # OpenCV's own threads can freeze inside forked workers
    torch.set_num_threads(1)


cv2.setNumThreads(0)
TIME_BUDGET_HOURS = 8.0       # Kaggle stops at 9 h; studies not reached by then get 0.5, so a file is always written
test_dl = DataLoader(TestSet(study_uids), batch_size=2, shuffle=False, num_workers=2,
                     worker_init_fn=worker_init, multiprocessing_context="fork")

models = {}
for name, fam in families.items():
    models[name] = []
    for path in fam["folds"]:
        m = KneeNetV5(fam["cfg"]["backbone"], pretrained=False).to(DEVICE).eval()
        state = torch.load(path, map_location=DEVICE)
        m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in state.items()})
        models[name].append(m)

# read each study once, run every model on it
logits = {name: {} for name in models}
fallback_batch_count, start = 0, time.time()
with torch.inference_mode():
    for batch_idx, (x, plane, pos, valid, uids) in enumerate(test_dl):
        if (time.time() - NOTEBOOK_START) / 3600 > TIME_BUDGET_HOURS:
            print(f"warning: time budget {TIME_BUDGET_HOURS} h reached; the remaining studies get 0.5")
            break
        try:
            x, plane, pos, valid = x.to(DEVICE), plane.to(DEVICE), pos.to(DEVICE), valid.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                for name, ms in models.items():
                    z = torch.stack([m(x, plane, pos, valid).float() for m in ms]).mean(0).cpu().numpy()
                    for uid, row in zip(uids, z):
                        logits[name][uid] = row
        except Exception as e:
            fallback_batch_count += 1
            print(f"warning: batch {batch_idx} failed, those studies get 0.5: {type(e).__name__}: {e}")
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        done = min((batch_idx + 1) * 2, len(study_uids))
        if batch_idx == 0 or done % 100 == 0 or done == len(study_uids):
            el = time.time() - start
            print(f"{done}/{len(study_uids)} studies | {el / 60:.1f} min | ETA {el / done * (len(study_uids) - done) / 60:.1f} min")

# folds were averaged inside each family; now average the families by rank (AUC only uses the order)
parts = []
for name in models:
    frame = pd.DataFrame([logits[name].get(u, np.full(len(LABELS), np.nan)) for u in study_uids], columns=LABELS)
    parts.append(frame.rank(pct=True))
predictions = np.nanmean(np.stack([p.values for p in parts]), axis=0)
predictions = np.nan_to_num(predictions, nan=0.5).astype("float32")
missing = sum(all(u not in logits[n] for n in models) for u in study_uids)
print("prediction rows:", len(predictions), " studies given 0.5:", missing, " failed batches:", fallback_batch_count)
if missing > 0.5 * len(study_uids):
    print("WARNING: more than half of the studies got no prediction. Check the warnings above before submitting.")
assert predictions.shape == (len(study_uids), len(LABELS))

2/3 studies | 0.1 min | ETA 0.1 min
3/3 studies | 0.1 min | ETA 0.0 min
prediction rows: 3  studies given 0.5: 0  failed batches: 0


/tmp/ipykernel_22/4229036394.py:70: RuntimeWarning: Mean of empty slice
  predictions = np.nanmean(np.stack([p.values for p in parts]), axis=0)


In [7]:
# Same write and check steps as our earlier submission cells, made safe for the hidden test run
def norm_name(name):
    return "".join(ch for ch in str(name).lower() if ch.isalnum())


# column names exactly as Kaggle writes them (from sample_submission.csv when it exists)
OUT_COLS = ["StudyInstanceUID"] + LABELS
sample_path = f"{COMP}/sample_submission.csv"
if os.path.isfile(sample_path):
    sample_cols = list(pd.read_csv(sample_path, nrows=1).columns)
    by_norm = {norm_name(col): col for col in sample_cols}
    if len(sample_cols) == len(OUT_COLS) and all(norm_name(col) in by_norm for col in OUT_COLS):
        OUT_COLS = [by_norm[norm_name(col)] for col in OUT_COLS]
    else:
        print("warning: sample_submission.csv columns do not match our 12 findings; using our names:", sample_cols)
print("submission columns:", OUT_COLS)


def write_submission(pred, studies, test_df, path):
    if len(pred) != len(studies):
        raise ValueError("prediction rows and StudyInstanceUID rows have different lengths")
    pred_frame = pd.DataFrame(pred, columns=LABELS).rank(pct=True, method="average")
    pred_frame.insert(0, "StudyInstanceUID", [str(x) for x in studies])
    sub = test_df[["StudyInstanceUID"]].copy()               # same rows and order as test.csv
    sub = sub.merge(pred_frame, on="StudyInstanceUID", how="left", validate="many_to_one")
    sub[LABELS] = sub[LABELS].fillna(0.5)
    sub.columns = OUT_COLS
    sub.to_csv(path, index=False)
    return sub


def validate_submission(path, test_df, tag="submission"):
    frame = pd.read_csv(path, dtype={OUT_COLS[0]: str})
    if list(frame.columns) != OUT_COLS:
        raise ValueError(f"{tag}: columns differ from the competition contract")
    if len(frame) != len(test_df):
        raise ValueError(f"{tag}: row count differs from test.csv")
    if list(frame[OUT_COLS[0]]) != list(test_df["StudyInstanceUID"]):
        raise ValueError(f"{tag}: StudyInstanceUID rows differ from test.csv")
    values = frame[OUT_COLS[1:]].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
    if not np.isfinite(values).all():
        raise ValueError(f"{tag}: non-finite prediction")
    if values.min() < 0.0 or values.max() > 1.0:
        raise ValueError(f"{tag}: prediction outside [0, 1]")
    return frame


os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")
write_submission(predictions, study_uids, test, "submission.csv")
validate_submission("submission.csv", test, "final")
assert os.path.getsize("submission.csv") > 0
print("FINAL CHECK PASSED", pd.read_csv("submission.csv").shape)

submission columns: ['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
FINAL CHECK PASSED (3, 13)
